# AXA Use Case 2: Finding Feature Interactions

Companion to `AXA_Use_Case_2_Mental_Health_Prediction.ipynb` (Section 6.3). This notebook runs on its own and answers one question: **do combinations of variables carry signal that a plain logistic regression misses, and if so, which ones?**

| Step | Question | Method |
|---|---|---|
| 1 | Do interactions matter at all? | Boosted trees of depth 1, 2 and 3 (depth 1 can only add up single-variable effects) |
| 2 | Which pairs interact? | Friedman's H-statistic on the log-odds scale |
| 3 | What does an interaction look like? | Predicted-risk heatmaps for the top pairs |
| 4 | Can the simple model use them? | Add the top pairs to logistic regression as cross-features |

**Protocol.** Stratified 60/20/20 train/validation/test split (seed 42). Everything is found on the training set and judged on the validation set. The **test set is scored once, at the very end.** The boosting settings and logistic-regression `C` values below come from the Bayesian optimisation in the main notebook (Section 5.2), so this notebook needs no tuning step.

**Requires:** `numpy pandas scikit-learn matplotlib seaborn`, and the repo's `src/` folder (this notebook is in `notebooks/`, so it finds `../src` itself).

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))     # repo root, so `src` imports
from src.data import load_csv, prepare, FEATURES, NUM, CAT
from src.models import make_linear, make_hgb, RANDOM_STATE
from src.evaluation import boot_auc_diff
from src.interactions import pairwise_h, make_bins, add_crosses

DATA_PATH = "depression_data.csv"          # <- change to where your Kaggle CSV lives
OUT_DIR = "interaction_outputs"; os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# ordered levels, used only to put heatmap rows and columns in a sensible order
ORDERS = {"Education Level": ["High School", "Associate Degree", "Bachelor's Degree", "Master's Degree", "PhD"],
          "Physical Activity Level": ["Sedentary", "Moderate", "Active"], "Alcohol Consumption": ["Low", "Moderate", "High"],
          "Dietary Habits": ["Unhealthy", "Moderate", "Healthy"], "Sleep Patterns": ["Poor", "Fair", "Good"],
          "Smoking Status": ["Non-smoker", "Former", "Current"]}

## 1. Data and split

In [ ]:
df = prepare(load_csv(DATA_PATH))
X, y = df[FEATURES], df["target"]
X_tr_full, X_te, y_tr_full, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr_full, y_tr_full, test_size=0.25, stratify=y_tr_full, random_state=RANDOM_STATE)
split_tab = pd.DataFrame({"rows": [len(y_tr), len(y_va), len(y_te)], "share_with_history": [y_tr.mean(), y_va.mean(), y_te.mean()]},
                         index=["train", "validation", "test"])
display(split_tab.style.format({"rows": "{:,}", "share_with_history": "{:.1%}"}))

## 2. Settings carried over from the main notebook (Bayesian optimisation, 5-fold CV)

In [ ]:
HGB_PARAMS = dict(learning_rate=0.0357, max_depth=8, max_leaf_nodes=49, min_samples_leaf=307, l2_regularization=0.0042)
C_ADDITIVE = 0.0531        # logistic regression with no interactions
C_INTERACT = 99.49         # logistic regression with cross-features (top of the search range, so treat as 'barely regularised')

## 3. Step 1: do interactions matter at all? (the depth ladder)

A boosted-tree model adds up many small trees. If every tree is a **stump** (depth 1, one split on one variable), the model can only be a sum of single-variable effects, which is exactly what a logistic regression assumes. Allowing depth 2 lets a tree split on a second variable under the first (a **two-way interaction**), and depth 3 allows three-way. If AUC does not rise with depth, there is nothing for interactions to add.

In [ ]:
rungs = [("Additive only (depth 1)", 1, 2), ("Up to 2-way (depth 2)", 2, 4), ("Up to 3-way (depth 3)", 3, 8),
         ("Tuned (depth 8)", HGB_PARAMS["max_depth"], HGB_PARAMS["max_leaf_nodes"])]
models, val_p, rows = {}, {}, []
for label, depth, leaves in rungs:
    m = make_hgb(NUM, CAT, **{**HGB_PARAMS, "max_depth": depth, "max_leaf_nodes": leaves}).fit(X_tr, y_tr)
    models[label] = m; val_p[label] = m.predict_proba(X_va)[:, 1]
    rows.append({"model": label, "val_auc": roc_auc_score(y_va, val_p[label])})
lin_add = make_linear(NUM, CAT, C_ADDITIVE).fit(X_tr, y_tr); val_p["Logistic (additive)"] = lin_add.predict_proba(X_va)[:, 1]
rows.append({"model": "Logistic regression (additive, for reference)", "val_auc": roc_auc_score(y_va, val_p["Logistic (additive)"])})
ladder = pd.DataFrame(rows); ladder.to_csv(os.path.join(OUT_DIR, "interaction_ladder.csv"), index=False)
display(ladder.style.format({"val_auc": "{:.4f}"}))

for a, b in [("Up to 2-way (depth 2)", "Additive only (depth 1)"), ("Up to 3-way (depth 3)", "Up to 2-way (depth 2)")]:
    d, lo, hi = boot_auc_diff(y_va, val_p[a], val_p[b])
    print(f"{a}  minus  {b}:  {d:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(["depth 1\n(additive)", "depth 2\n(pairs)", "depth 3\n(triples)"], ladder.val_auc[:3], color=["#4A566E", "#1F7A7A", "#1F7A7A"])
for i, v in enumerate(ladder.val_auc[:3]): ax.text(i, v + .0002, f"{v:.4f}", ha="center", fontweight="bold")
ax.set_ylim(ladder.val_auc[:3].min() - .003, ladder.val_auc[:3].max() + .002); ax.set_ylabel("validation ROC-AUC"); ax.set_title("Allowing combinations of variables")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "depth_ladder.png")); plt.show()

**How to read it.** The step from depth 1 to depth 2 is the benefit of two-way interactions. The step from depth 2 to depth 3 is the extra benefit of three-way interactions. If the bootstrap interval for a step includes 0, treat that step as noise.

## 4. Step 2: which pairs interact? (Friedman's H-statistic)

For a pair of variables A and B, using the tuned boosted model:

1. Compute the model's **joint effect** of A and B on the **log-odds** scale (average prediction while forcing A and B to each person's values).
2. Compute the effect of A alone and of B alone the same way.
3. **Interaction = joint effect − effect of A − effect of B.** If the model were additive in A and B, this would be 0 everywhere.
4. **Strength** = root-mean-square size of that leftover, in log-odds units. The column `h2` is the classic relative version (leftover variance as a share of the joint effect's variance).

I rank by `strength` because `h2` is unstable for pairs whose joint effect is tiny: it divides by a number close to zero and can look large for pairs that barely matter.

In [ ]:
S_h = X_tr.sample(300, random_state=RANDOM_STATE)           # 300 training people: the calculation is 300 x 300 predictions per pair
h_table = pairwise_h(models["Tuned (depth 8)"], S_h, FEATURES)
h_table["pair"] = h_table.feature_a + "  x  " + h_table.feature_b
h_table.to_csv(os.path.join(OUT_DIR, "interaction_strength.csv"), index=False)
display(h_table.head(10)[["pair", "strength", "h2"]].style.format({"strength": "{:.4f}", "h2": "{:.3f}"}))

top = h_table.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(top.pair, top.strength, color=["#E0A030" if i >= len(top) - 3 else "#CFCBC0" for i in range(len(top))])
ax.set_xlabel("interaction strength (log-odds; 0 = purely additive)"); ax.set_title("Strongest pairwise interactions")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "interaction_strength.png")); plt.show()

**Caution.** H-statistics are noisy for weak variables and depend on the 300-person sample. Treat the top few as candidates, and keep a pair only if it improves **validation** AUC (Step 4).

## 5. Step 3: what does an interaction look like?

Each heatmap forces two variables to every combination, predicts risk with the tuned boosted model and averages over 3,000 training people. If effects were purely additive, **every row would show the same left-to-right pattern**. Rows that differ are the interaction.

In [ ]:
S_g = X_tr.sample(3000, random_state=RANDOM_STATE)
def grid_values(col):
    if col in NUM: return list(np.unique(np.round(S_g[col].quantile([.1, .3, .5, .7, .9]).values, 2)))
    present = set(S_g[col]); return [v for v in ORDERS.get(col, sorted(present)) if v in present]
def risk_grid(model, a, b):
    ga, gb = grid_values(a), grid_values(b); out = pd.DataFrame(index=ga, columns=gb, dtype=float)
    for va in ga:
        for vb in gb:
            T = S_g.copy(); T[a] = va; T[b] = vb; out.loc[va, vb] = model.predict_proba(T)[:, 1].mean()
    return out
pairs_top = [(r.feature_a, r.feature_b) for r in h_table.head(3).itertuples()]
grids = [risk_grid(models["Tuned (depth 8)"], a, b) for a, b in pairs_top]
vmin, vmax = min(g.values.min() for g in grids), max(g.values.max() for g in grids)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, g, (a, b) in zip(axes, grids, pairs_top):
    sns.heatmap(g, annot=True, fmt=".0%", cmap="YlOrBr", vmin=vmin, vmax=vmax, cbar=False, ax=ax)
    ax.set_xlabel(b); ax.set_ylabel(a); ax.set_title(f"{a} x {b}")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "interaction_heatmaps.png")); plt.show()

## 6. Step 4: give the interactions to the logistic regression

Each chosen pair becomes one **cross-feature**: a single categorical variable holding the combined category (numeric variables are cut into quartiles first, with cut-points learned on training data only). One-hot encoding then lets the logistic regression learn a separate effect for every combination.

How many pairs (K)? We add the ranked pairs one at a time and keep the **smallest K within 0.0005 of the best validation AUC**, so we stop adding terms once they stop helping.

In [ ]:
bins = make_bins(X_tr, NUM)
cand_pairs = list(zip(h_table.feature_a, h_table.feature_b))[:6]
def fit_cross_logit(K, C):
    Xa, names = add_crosses(X_tr, cand_pairs[:K], bins)
    return make_linear(NUM, CAT + names, C).fit(Xa, y_tr), names

k_rows = []
for K in range(0, len(cand_pairs) + 1):
    m, names = fit_cross_logit(K, C_INTERACT)
    Xv, _ = add_crosses(X_va, cand_pairs[:K], bins)
    k_rows.append({"pairs added (K)": K, "added": ", ".join(names) if names else "-", "val_auc": roc_auc_score(y_va, m.predict_proba(Xv)[:, 1])})
k_tab = pd.DataFrame(k_rows); display(k_tab.style.format({"val_auc": "{:.4f}"}))
K_best = int(min(r["pairs added (K)"] for r in k_rows if r["val_auc"] >= k_tab.val_auc.max() - 0.0005))
print(f"Chosen K = {K_best}")

## 7. Final comparison (test set, scored once)

In [ ]:
lin_int, names_best = fit_cross_logit(K_best, C_INTERACT)
Xte_i, _ = add_crosses(X_te, cand_pairs[:K_best], bins); Xv_i, _ = add_crosses(X_va, cand_pairs[:K_best], bins)
pt = {"Logistic regression (additive)": lin_add.predict_proba(X_te)[:, 1],
      f"Logistic regression + {K_best} interaction(s)": lin_int.predict_proba(Xte_i)[:, 1],
      "Gradient boosting (tuned)": models["Tuned (depth 8)"].predict_proba(X_te)[:, 1]}
pv = {"Logistic regression (additive)": val_p["Logistic (additive)"],
      f"Logistic regression + {K_best} interaction(s)": lin_int.predict_proba(Xv_i)[:, 1],
      "Gradient boosting (tuned)": val_p["Tuned (depth 8)"]}
cmp_tab = pd.DataFrame([{"model": k, "val_auc": roc_auc_score(y_va, pv[k]), "test_auc": roc_auc_score(y_te, pt[k])} for k in pt])
display(cmp_tab.style.format({"val_auc": "{:.4f}", "test_auc": "{:.4f}"})); cmp_tab.to_csv(os.path.join(OUT_DIR, "interaction_model_comparison.csv"), index=False)
names = list(pt)
d1 = boot_auc_diff(y_te, pt[names[1]], pt[names[0]]); d2 = boot_auc_diff(y_te, pt[names[2]], pt[names[1]])
print(f"Test AUC, interaction logistic minus additive logistic: {d1[0]:+.4f}  95% CI [{d1[1]:+.4f}, {d1[2]:+.4f}]")
print(f"Test AUC, gradient boosting minus interaction logistic: {d2[0]:+.4f}  95% CI [{d2[1]:+.4f}, {d2[2]:+.4f}]")

In [ ]:
# readable coefficients: odds ratios for the cross-feature terms (net effect needs the main effects too, so read the pattern, not each number)
if K_best > 0:
    pre = lin_int.named_steps["pre"]; oh = pre.named_transformers_["cat"].named_steps["oh"]
    try: cn = list(oh.get_feature_names_out(CAT + names_best))
    except AttributeError: cn = list(oh.get_feature_names(CAT + names_best))
    num_cols = list(pre.transformers_[0][2])
    co = pd.DataFrame({"term": num_cols + cn, "odds_ratio": np.exp(lin_int.named_steps["clf"].coef_[0])})
    co = co[co.term.str.contains(" x ")]
    co = co.reindex((co.odds_ratio - 1).abs().sort_values(ascending=False).index).head(10)
    display(co.style.format({"odds_ratio": "{:.3f}"})); co.to_csv(os.path.join(OUT_DIR, "interaction_odds_ratios.csv"), index=False)

## 8. What to take from this

* **Pairs matter a little, triples do not.** Check the depth-ladder intervals above: the step from depth 1 to 2 is the one that should stand apart from 0, and depth 3 should add nothing.
* **The interacting variables are the ones that drive the outcome anyway** (employment, education, smoking), so interactions refine the same story rather than adding a new one.
* **The simple model can keep up.** Adding the top cross-features to logistic regression should close most of the gap to boosting while staying explainable.
* **Caveats.** The pairs were found on this one dataset, so confirm them on fresh data. The gain in AUC is small. The dataset is synthetic, so patterns such as smoking raising risk for the employed but lowering it for the unemployed are features of how it was generated, not health findings.
* **Optional extension.** The main notebook (Section 6.3) cross-checks this ranking with SHAP interaction values from XGBoost (`pip install xgboost shap`).